# **Deeploc 2.1**

2025 겨울 URP / 문서연, 김대현, 권효재


---


여기서부턴 Deeploc 2.1과 같은 구조로 학습을 진행!


---


개선/공부 필요(2026_01_18):

---

# **라이브러리 설명**

# torch
Pytorch 라이프러리 패키지


* torch.autograd: 자동 미분을 위한 함수가 포함됨(ex. enable/no_grad: 자동 미분 on/off, Function: 자체 미분 함수 정의 클래스)
* torch.nn: 신경망 구축을 위한 기본 데이터 구조/레이어(RNN/LSTM)/활성화 함수(ReLU)/손실 함수(MSELoss) 포함됨
* torch.optim: 확률적 경사 하강법(Stochastic Gradient Descent, SGD) 중심의 파라미터 옵티마이저 알고리즘
* torch.utils.data: SDG 반복연산 시에 사용하는 미니배치 유틸리티 포함됨
* torch.onnx: ONNX(Open Neural Network Exchange) 포맷으로 모델 export 할 때 사용

In [3]:
#필요 라이브러리 설치 및 import
!pip install -q torch pandas numpy safetensors

from safetensors import safe_open
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import sigmoid_focal_loss as FocalLoss

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import math

In [5]:
#<저장소 설정>

#Colab 환경
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/Github/Mprotein_hydrophobic/ESMC_embedding_safetensors'

#로컬 환경
#SAVE_PATH = './ESMC_embedding'

os.makedirs(SAVE_PATH, exist_ok=True)

Mounted at /content/drive


In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [16]:
testtensor = 'embeddings_part_3.safetensors'
testtarget = 'target_part_3.pt'
p1 = os.path.join(SAVE_PATH, testtensor)
p2 = os.path.join(SAVE_PATH, testtarget)
#for k in range(4):
#  SAVE_PATH_EMBEDDINGS = f'embeddings_part_{k}.safetensors'
#  SAVE_PATH_TARGETS = f'targets_part_{k}.pt'
with safe_open(p1, framework="pt", device="cpu") as f:
  a = f.keys()
  print(a)
  b = f.get_tensor('A0A023W140')
  print(b)
  print(b.shape)

targetsmp = torch.load(p2, weights_only=False)
target = targetsmp['A0A023W140']
print(target)

'''class Dataset(Dataset):
  #데이터셋 전처리
  def __init__(self):
    x_train
    y_
    Y_VAILDATIO

  #데이터셋 길이 / 샘플 길이
  def __len__(self):
    return(len(self.x_train)

  #데이터셋 샘플 1개 가져오기
  def __getitem__(self, idx):'''

['A0A023W140', 'A0A023W145', 'A0A059T2H4', 'A0A060A682', 'A0A060L102', 'A0A060L4I9', 'A0A061ACU2', 'A0A067CMC7', 'A0A067XR63', 'A0A067XRK9', 'A0A067YMX8', 'A0A075TR27', 'A0A075TRK9', 'A0A088MLT8', 'A0A0A0LLY1', 'A0A0B5A8P4', 'A0A0B5AC95', 'A0A0B5KYT4', 'A0A0B5L781', 'A0A0C5CJR8', 'A0A0D1CD96', 'A0A0E3SVE7', 'A0A0E3T552', 'A0A0F7YYV0', 'A0A0F7YYX3', 'A0A0G2JXN2', 'A0A0G2KTI4', 'A0A0H2XIV9', 'A0A0H2ZNG3', 'A0A0H3AKU6', 'A0A0H3CDY2', 'A0A0H3GGE2', 'A0A0H3H7Y9', 'A0A0H3K686', 'A0A0H3KDT7', 'A0A0H3MGR4', 'A0A0H3MGR5', 'A0A0M5K865', 'A0A0P0VG31', 'A0A0P0XM10', 'A0A0P1ATX0', 'A0A0P1B5Z9', 'A0A0R4IM31', 'A0A0S2UWC9', 'A0A0S4FKT4', 'A0A0U1QT59', 'A0A0U2X0E4', 'A0A125S9E1', 'A0A125S9E2', 'A0A125S9F3', 'A0A125S9F9', 'A0A160PB22', 'A0A172M418', 'A0A172M419', 'A0A172M477', 'A0A172M485', 'A0A172M490', 'A0A172M4M8', 'A0A182BSR9', 'A0A182BSS2', 'A0A193AU77', 'A0A193AUF6', 'A0A1B1FHP3', 'A0A1D6KL43', 'A0A1D8PK89', 'A0A1D8PMB1', 'A0A1D8PN88', 'A0A1D8PNP3', 'A0A1D8PPI5', 'A0A1E1WVX2', 'A0A1F4', 'A0A1I9KN

In [8]:
#앞에서 Layernorm 해준걸 인풋으로 받는다고 가정
class MultiheadAttentionPooling(nn.Module):
  def __init__(self,
              attn_dim = 128,      #hyperparameter
              heads_nums = 2,       #hyperparameter
              kernel_size = 5,     #hyperparameter
              ):
    super().__init__()

    self.attn_dim = attn_dim
    #우선은,, 멀티헤드로 구현을 한다
    self.heads_nums = heads_nums
    #멀티헤드의 디멘션은 전체 디멘션을 헤드 개수로 나눈것
    #이거때문에 헤드 개수를 잘 나눠지도록(?) 설정함 보통
    self.head_dim = attn_dim // heads_nums

    # Q, K = V 만들기
    # Q : learneable Query, 먼저 (1, 1, attn_dim) 에 해당하는 빈 벡터 > xavier_uniform_ 하면 입출력 고려해서 난수생성 가능
    self.query = nn.Parameter(torch.empty(1, 1, attn_dim))
    nn.init.xavier_uniform_(self.query.data)
    self.w_kv = nn.Linear(attn_dim, attn_dim)                 #입력/출력 크기가 attn_dim인 Linear FC를 수행하는 모듈 // key value 짜피 같으니까 이걸로 한번에 할거고 논문도 그렇게 했는데 둘이 따로 초기화한다면? 즉 파라미터가 두개라면..?

  def forward(self, x):
    #입력으로 받을 형태: (batch_size, sequence_length, attn_dim)
    batch_size, sequence_length, _ = x.shape


    #배치 사이즈에 맞게 복제: 각 배치 샘플 전부 같은 쿼리 파라미터 공유
    #attn_dim을 헤드별로 쪼갬()
    #헤드별로 계산할거라서 헤드를 앞으로 뺌
    Q = self.query.repeat(batch_size, 1, 1).view(batch_size, 1, self.heads_nums, self.head_dim).transpose(1, 2)
    K = self.w_kv(x).view(batch_size, sequence_length, self.heads_nums, self.head_dim).transpose(1, 2)
    V = self.w_kv(x).view(batch_size, sequence_length, self.heads_nums, self.head_dim).transpose(1, 2)
    # 셋다 (batch_size, head_nums, sequence_length(Q:1; per token), head_dim)
    #print(Q)
    #print(K)
    #print(V)
    #행렬곱 > Attention Scalar score 구함!
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
    print(f'어텐션 Score: {scores.shape}')
    #Conv1d 가우시안 필터
    scores_c1d_gaussian = scores

    #Softmax 적용
    attn_weights = F.softmax(scores_c1d_gaussian, dim=-1)
    print(f'어텐션 Weights: {attn_weights.shape}')

    #행렬곱 > weighted attention
    attnpooled_nosum = torch.matmul(attn_weights, V)
    print(f'어텐션 Pooling (합하기 전): {attnpooled_nosum.shape}')

    #가중합을 위해 (batch_size, head_nums, 1, head_dim) 순서니까
    #1 없애고 reshape로 head concat해주기
    attentionpooled = attnpooled_nosum.squeeze(2).reshape(batch_size, self.attn_dim)
    print(f'어텐션 Pooling: {attentionpooled.shape}')

    return attentionpooled

In [9]:
linearfc_embdtoattn = nn.Linear(1152, 128)
b_linear = linearfc_embdtoattn(b)
print(b_linear.shape)

torch.Size([1, 88, 128])


In [10]:
testpooling = MultiheadAttentionPooling(128, 2)
testpooled = testpooling.forward(b_linear)
print(testpooled.shape)

어텐션 Score: torch.Size([1, 2, 1, 88])
어텐션 Weights: torch.Size([1, 2, 1, 88])
어텐션 Pooling (합하기 전): torch.Size([1, 2, 1, 64])
어텐션 Pooling: torch.Size([1, 128])
torch.Size([1, 128])


In [23]:
linearfc_attntoclass = nn.Linear(128, 4)
testclass = linearfc_attntoclass(testpooled)
print(testclass.shape)
target_ = torch.from_numpy(target).float()
print(target_.shape)

torch.Size([1, 4])
torch.Size([1, 4])


In [24]:
loss = FocalLoss(testclass, target_, alpha = 0.25, gamma = 2, reduction = 'mean')
print(loss)

tensor(8.6814, grad_fn=<MeanBackward0>)


In [ ]:
class model(nn.Module):
  def __init__(self):
    super().__init__()




  def forward(self, x):
    return x
